# Fase 6 — Deployment | CardioRisk
**IBM Data Science Professional Certificate · CRISP-DM**

EWS (Early Warning Score) alineado con Business Understanding (Fase 1):
- Bajo Riesgo: p < 0.30 → Monitoreo estándar
- Riesgo Medio: 0.30 ≤ p ≤ 0.65 → Vigilancia intensiva
- Alto Riesgo: p > 0.65 → Alerta inmediata UCI

In [ ]:
# BLOQUE 0 — Instalaciones e imports
!pip install -q kagglehub scikit-learn imbalanced-learn joblib
import os, warnings, joblib
import pandas as pd
import numpy as np
import matplotlib, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (recall_score, precision_score, f1_score,
    roc_auc_score, accuracy_score, confusion_matrix, roc_curve)
from imblearn.over_sampling import SMOTE

# Umbrales EWS — definidos en Fase 1 Business Understanding
EWS_BAJO  = 0.30   # p < 0.30  → Bajo Riesgo   (Monitoreo estándar)
EWS_MEDIO = 0.65   # p ≤ 0.65  → Riesgo Medio  (Vigilancia intensiva)
#                    p > 0.65  → Alto Riesgo   (Alerta inmediata UCI)

COLOR_BG='#0a0f1a'; COLOR_AX='#0d1526'; COLOR_TEXT='#e2e8f0'; COLOR_GRID='#1a2c3d'
matplotlib.rcParams.update({'figure.facecolor':COLOR_BG,'axes.facecolor':COLOR_AX,
    'axes.edgecolor':COLOR_GRID,'axes.labelcolor':COLOR_TEXT,
    'xtick.color':'#7a8fa8','ytick.color':'#7a8fa8',
    'grid.color':COLOR_GRID,'text.color':COLOR_TEXT})
print('✅ Entorno listo')
print(f'EWS umbrales: BAJO<{EWS_BAJO} | MEDIO {EWS_BAJO}-{EWS_MEDIO} | ALTO>{EWS_MEDIO}')

In [ ]:
# BLOQUE 1 — Pipeline completo final
print('='*60); print('PIPELINE FASE 6 — DEPLOYMENT CARDIORISK'); print('='*60)
try:
    import kagglehub
    path = kagglehub.dataset_download('jocelyndumlao/cardiovascular-disease-dataset')
    csv_path = next(os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs if f.endswith('.csv'))
    df = pd.read_csv(csv_path); print(f'✅ KaggleHub: {csv_path}')
except Exception:
    df = pd.read_csv('/content/cardiovascular_disease_dataset.csv'); print('✅ CSV local')
df.columns = df.columns.str.lower().str.strip(); df = df.dropna()
print(f'Shape: {df.shape}')
TARGET='target'; CATEGORICAL=['gender','chestpain','restingrelectro']
df_enc = pd.get_dummies(df, columns=CATEGORICAL, drop_first=True)
# CORRECCIÓN: excluir patientid (ID administrativo sin valor clínico)
feature_cols = [c for c in df_enc.columns if c not in [TARGET,'patientid']]
print(f'✅ patientid excluido. Features: {len(feature_cols)}')
print(f'OHE cols: {[c for c in feature_cols if any(c.startswith(cat) for cat in CATEGORICAL)]}')
X = df_enc[feature_cols]; y = df_enc[TARGET]
X_temp,X_test,y_temp,y_test = train_test_split(X,y,test_size=0.15,stratify=y,random_state=42)
X_train,X_val,y_train,y_val = train_test_split(X_temp,y_temp,test_size=0.1765,stratify=y_temp,random_state=42)
print(f'Train:{X_train.shape[0]} Val:{X_val.shape[0]} Test:{X_test.shape[0]}')
scaler=StandardScaler(); X_train_sc=scaler.fit_transform(X_train)
X_val_sc=scaler.transform(X_val); X_test_sc=scaler.transform(X_test)
smote=SMOTE(random_state=42); X_train_sm,y_train_sm=smote.fit_resample(X_train_sc,y_train)
model=GradientBoostingClassifier(learning_rate=0.1,max_depth=5,n_estimators=200,random_state=42)
model.fit(X_train_sm,y_train_sm); print('✅ GradientBoostingClassifier entrenado')
joblib.dump(model,'/content/cardiorisk_model.pkl')
joblib.dump(scaler,'/content/cardiorisk_scaler.pkl')
joblib.dump(feature_cols,'/content/cardiorisk_features.pkl')
print('✅ Artefactos guardados'); print('='*60)

In [ ]:
# BLOQUE 2 — Métricas TEST SET
y_pred=model.predict(X_test_sc); y_pred_proba=model.predict_proba(X_test_sc)[:,1]
recall=recall_score(y_test,y_pred); precision=precision_score(y_test,y_pred)
f1=f1_score(y_test,y_pred); auc=roc_auc_score(y_test,y_pred_proba)
accuracy=accuracy_score(y_test,y_pred); cm_matrix=confusion_matrix(y_test,y_pred)
tn,fp,fn,tp=cm_matrix.ravel()
print('='*60); print('EVALUACIÓN FINAL — TEST SET (EWS CardioRisk)'); print('='*60)
print(f'  Recall    : {recall:.4f}  ← KPI primario (Error tipo II = FN/{tp+fn} = {fn/(tp+fn):.1%})')
print(f'  Precision : {precision:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  AUC-ROC   : {auc:.4f}  ← KPI secundario (objetivo > 0.85)')
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  TP={tp} TN={tn} FP={fp} FN={fn}')
print()
kpi1 = recall >= 0.80
kpi2 = auc > 0.85
kpi3 = fn/(tp+fn) < 0.05
print(f'  KPI Recall>0.80:         {"✅" if kpi1 else "❌"} {recall:.4f}')
print(f'  KPI AUC-ROC>0.85:        {"✅" if kpi2 else "❌"} {auc:.4f}')
print(f'  KPI Error tipo II <5%:   {"✅" if kpi3 else "❌"} {fn/(tp+fn):.1%}')

In [ ]:
# BLOQUE 3 — EWS: función deployable
# Early Warning Score alineado con Business Understanding (Fase 1)
print('='*60)
print('EWS — Early Warning Score CardioRisk')
print(f'  Bajo Riesgo  : p < {EWS_BAJO}  → Monitoreo estándar')
print(f'  Riesgo Medio : {EWS_BAJO} ≤ p ≤ {EWS_MEDIO} → Vigilancia intensiva')
print(f'  Alto Riesgo  : p > {EWS_MEDIO}  → Alerta inmediata UCI')
print('='*60)

# Verificar columnas OHE generadas
ohe_cols = [c for c in feature_cols if any(c.startswith(cat+'_') for cat in CATEGORICAL)]
print('Columnas OHE en el pipeline:', ohe_cols)


def predecir_riesgo_cardiovascular(paciente: dict, modelo=model, sc=scaler, cols=feature_cols) -> dict:
    '''
    EWS (Early Warning Score) CardioRisk.
    Estratifica el riesgo isquémico en tiempo real según umbrales F1.

    Args:
        paciente: dict con variables clínicas numéricas (sin patientid).
                  gender: 0/1, chestpain: 0/1/2/3, restingrelectro: 0/1/2
    Returns:
        dict: riesgo(0/1), probabilidad, ews_nivel, accion_clinica
    '''
    df_p = pd.DataFrame([paciente])
    df_p.columns = df_p.columns.str.lower().str.strip()
    # OHE manual — reproduce drop_first=True sobre valores numéricos del dataset
    for cat in CATEGORICAL:
        if cat in df_p.columns:
            val = int(df_p[cat].iloc[0])
            unique_vals = sorted([int(str(c).replace(cat+'_','')) for c in cols if c.startswith(cat+'_')])
            for v in unique_vals:
                df_p[f'{cat}_{v}'] = 1 if val == v else 0
            df_p = df_p.drop(columns=[cat])
    for col in cols:
        if col not in df_p.columns:
            df_p[col] = 0
    df_p = df_p[cols]
    proba = float(modelo.predict_proba(sc.transform(df_p.values))[0,1])
    pred  = 1 if proba >= 0.5 else 0

    # Estratificación EWS — umbrales de Fase 1
    if proba < EWS_BAJO:
        nivel  = 'Bajo Riesgo'
        accion = 'Monitoreo estándar. Control ambulatorio. Hábitos saludables.'
    elif proba <= EWS_MEDIO:
        nivel  = 'Riesgo Medio'
        accion = 'Vigilancia intensiva. Observación 6-12h. Biomarcadores (troponina, BNP). ECG seriado.'
    else:
        nivel  = 'Alto Riesgo'
        accion = 'ALERTA INMEDIATA UCI. Activar protocolo SCA. Cardiólogo de guardia STAT.'

    return {'riesgo': pred, 'probabilidad': round(proba,4),
            'ews_nivel': nivel, 'accion_clinica': accion}


# 3 pacientes de prueba
casos = [
    ('Mujer 35a — perfil saludable',
     {'age':35,'restingbp':118,'serumcholestrol':175,'maxheartrate':155,
      'oldpeak':0.3,'noofmajorvessels':0,'fastingbloodsugar':0,
      'exerciseangia':0,'slope':2,'gender':0,'chestpain':0,'restingrelectro':0}),
    ('Hombre 58a — riesgo medio',
     {'age':58,'restingbp':140,'serumcholestrol':245,'maxheartrate':130,
      'oldpeak':1.2,'noofmajorvessels':1,'fastingbloodsugar':1,
      'exerciseangia':0,'slope':1,'gender':1,'chestpain':2,'restingrelectro':1}),
    ('Hombre 67a — alto riesgo UCI',
     {'age':67,'restingbp':162,'serumcholestrol':285,'maxheartrate':108,
      'oldpeak':3.6,'noofmajorvessels':3,'fastingbloodsugar':1,
      'exerciseangia':1,'slope':0,'gender':1,'chestpain':3,'restingrelectro':2}),
]
for nombre,datos in casos:
    r=predecir_riesgo_cardiovascular(datos)
    print(f'\nPaciente: {nombre}')
    print(f'  EWS Nivel:  {r["ews_nivel"]}  (p={r["probabilidad"]:.1%})')
    print(f'  Acción:     {r["accion_clinica"]}')
print('\n✅ EWS deployable OK')

In [ ]:
# BLOQUE 4 — Dashboard EWS final
fpr_arr,tpr_arr,_=roc_curve(y_test,y_pred_proba)
imp_series=pd.Series(model.feature_importances_,index=feature_cols).sort_values().tail(8)

fig=plt.figure(figsize=(14,10),facecolor=COLOR_BG)
# 4 KPI cards con posición absoluta
kpi_data=[('Recall',f'{recall:.4f}','#00f2fe','KPI primario: >0.80'),
          ('AUC-ROC',f'{auc:.4f}','#8b5cf6','KPI secundario: >0.85'),
          ('Error II',f'{fn/(tp+fn):.1%}','#00ff88','KPI: <5% (FN)'),
          ('FN',str(fn),'#ff3355','Alertas UCI perdidas')]
for i,(label,val,color,desc) in enumerate(kpi_data):
    ax=fig.add_axes([0.03+i*0.245, 0.72, 0.22, 0.24])
    ax.set_facecolor(COLOR_AX); ax.axis('off')
    ax.add_patch(FancyBboxPatch((0.05,0.05),0.9,0.9,boxstyle='round,pad=0.03',
                facecolor='#121a2e',edgecolor=color,linewidth=2.0))
    ax.text(0.5,0.67,val,ha='center',va='center',fontsize=26,fontweight='bold',color=color)
    ax.text(0.5,0.38,label,ha='center',va='center',fontsize=12,color=COLOR_TEXT)
    ax.text(0.5,0.15,desc,ha='center',va='center',fontsize=8,color='#7a8fa8')
fig.text(0.5,0.975,'CardioRisk EWS — Dashboard Clínico Final',
         ha='center',fontsize=18,fontweight='bold',color=COLOR_TEXT)

gs=gridspec.GridSpec(2,3,figure=fig,left=0.07,right=0.97,
                     top=0.68,bottom=0.08,hspace=0.45,wspace=0.32)

# Matriz de confusión
ax_cm=fig.add_subplot(gs[0,0])
ax_cm.imshow(cm_matrix,cmap='Blues',aspect='auto',alpha=0.8)
lbs=[['TN','FP'],['FN','TP']]
for i in range(2):
    for j in range(2):
        clr='#ff3355' if (i==1 and j==0) else COLOR_TEXT
        ax_cm.text(j,i,f'{cm_matrix[i,j]}\n({lbs[i][j]})',ha='center',va='center',
                   fontsize=13,fontweight='bold',color=clr)
ax_cm.set_xticks([0,1]); ax_cm.set_yticks([0,1])
ax_cm.set_xticklabels(['Pred.Neg','Pred.Pos']); ax_cm.set_yticklabels(['Real Neg','Real Pos'])
ax_cm.set_title('Matriz de Confusión',fontweight='bold',pad=8)

# Curva ROC
ax_roc=fig.add_subplot(gs[0,1])
ax_roc.plot(fpr_arr,tpr_arr,color='#00ff88',lw=2.5,label=f'AUC={auc:.4f}')
ax_roc.plot([0,1],[0,1],'--',color='#3a5570',lw=1)
ax_roc.fill_between(fpr_arr,tpr_arr,alpha=0.1,color='#00ff88')
ax_roc.set_title('Curva ROC',fontweight='bold',pad=8)
ax_roc.set_xlabel('Tasa FP'); ax_roc.set_ylabel('Tasa VP (Recall)')
ax_roc.legend(loc='lower right',framealpha=0.7); ax_roc.grid(True,alpha=0.2)
ax_roc.set_xlim([-0.02,1.02]); ax_roc.set_ylim([-0.02,1.02])

# Feature importances
ax_fi=fig.add_subplot(gs[0,2])
bc=['#ff3355' if any(k in n for k in ['slope','chest','oldpeak']) else '#8b5cf6' for n in imp_series.index]
bars=ax_fi.barh(range(len(imp_series)),imp_series.values,color=bc,alpha=0.85)
ax_fi.set_yticks(range(len(imp_series))); ax_fi.set_yticklabels(imp_series.index,fontsize=9)
ax_fi.set_title('Top 8 Predictores (Gini)',fontweight='bold',pad=8)
ax_fi.set_xlabel('Importancia'); ax_fi.grid(True,alpha=0.2,axis='x')
for bar,val in zip(bars,imp_series.values):
    ax_fi.text(bar.get_width()+0.003,bar.get_y()+bar.get_height()/2,f'{val:.3f}',va='center',fontsize=8)

# Panel EWS
ax_txt=fig.add_subplot(gs[1,:])
ax_txt.set_facecolor('#0d1526'); ax_txt.axis('off')
ews_txt=(f'EWS — ESTRATIFICACIÓN DE RIESGO (alineado con Business Understanding Fase 1)\n'
    f'● Bajo Riesgo  (p < {EWS_BAJO}):            Monitoreo estándar\n'
    f'● Riesgo Medio ({EWS_BAJO} ≤ p ≤ {EWS_MEDIO}):  Vigilancia intensiva — observación 6-12h, biomarcadores\n'
    f'● Alto Riesgo  (p > {EWS_MEDIO}):            ALERTA INMEDIATA UCI — protocolo SCA\n\n'
    f'Test set: TP={tp} | TN={tn} | FP={fp} | FN={fn}  —  '
    f'Error tipo II = {fn/(tp+fn):.1%} (KPI: <5%)')
ax_txt.text(0.5,0.5,ews_txt,ha='center',va='center',fontsize=9.5,color=COLOR_TEXT,linespacing=1.8,
            bbox=dict(boxstyle='round,pad=0.7',facecolor='#121a2e',edgecolor='#1a2c3d',alpha=0.9))

plt.savefig('/content/f6_dashboard_final.png',dpi=150,bbox_inches='tight',facecolor=COLOR_BG)
plt.show(); print('✅ Dashboard guardado: /content/f6_dashboard_final.png')

In [ ]:
# BLOQUE 5 — Reporte CRISP-DM completo
s1='═'*70; s2='─'*70
print(f'\n{s1}'); print(' '*20+'REPORTE FINAL CRISP-DM — CARDIORISK')
print(' '*20+'IBM Data Science Professional Certificate'); print(s1)

print(f'\n{s2}\n1. BUSINESS UNDERSTANDING\n{s2}')
print('  Obj. Negocio: Reducir mortalidad hospitalaria 20% y optimizar rotación UCI')
print('                mediante altas tempranas seguras.')
print('  Obj. DS:      Clasificador con EWS auditable para estratificar riesgo')
print('                isquémico en tiempo real. Salida binaria (0/1) mapeada a')
print('                3 niveles EWS via probabilidad calibrada del modelo.')
print('  KPIs:         Recall > 80% en Alto Riesgo | AUC-ROC > 0.85 | Error II < 5%')

print(f'\n{s2}\n2. DATA UNDERSTANDING\n{s2}')
print(f'  Dataset:  jocelyndumlao/cardiovascular-disease-dataset (KaggleHub)')
print(f'  N={len(df)} | Cols={len(df.columns)} | Target={dict((df[TARGET].value_counts(normalize=True)*100).round(1))}')
print('  Fix:      patientid excluido (ID administrativo sin valor clínico).')

print(f'\n{s2}\n3. DATA PREPARATION\n{s2}')
print(f'  Features: {len(feature_cols)} tras OHE y exclusión de patientid')
print('  OHE: gender, chestpain, restingrelectro (drop_first=True)')
print('  Split: 70/15/15 estratificado | Scaler fit SOLO en train | SMOTE SOLO en train')

print(f'\n{s2}\n4. MODELING\n{s2}')
print('  1.LogReg Recall=0.955 AUC=0.984 | 2.SVM Recall=0.955 AUC=0.992')
print('  3.RF Recall=0.966 AUC=0.995 | 4.XGB Recall=0.966 AUC=0.995')
print('  5.GradientBoosting Recall=1.000 AUC=0.997 ← GANADOR')
print('  Hiperparámetros: learning_rate=0.1 | max_depth=5 | n_estimators=200')

print(f'\n{s2}\n5. EVALUATION — TEST SET SELLADO\n{s2}')
print(f'  Recall={recall:.4f}   KPI Recall>0.80: {"✅" if recall>=0.80 else "❌"}')
print(f'  AUC-ROC={auc:.4f} KPI AUC>0.85:    {"✅" if auc>0.85 else "❌"}')
print(f'  Error II={fn/(tp+fn):.1%}  KPI Error II<5%: {"✅" if fn/(tp+fn)<0.05 else "❌"}')
print(f'  F1={f1:.4f} | Precision={precision:.4f} | Accuracy={accuracy:.4f}')
print(f'  TP={tp} TN={tn} FP={fp} FN={fn}')

print(f'\n{s2}\n6. DEPLOYMENT — EWS CardioRisk\n{s2}')
print('  Artefactos:')
print('    cardiorisk_model.pkl | cardiorisk_scaler.pkl | cardiorisk_features.pkl')
print()
print('  predecir_riesgo_cardiovascular(paciente:dict) -> dict')
print('    Input:  variables clínicas numéricas (sin patientid)')
print('    Output: riesgo(0/1), probabilidad, ews_nivel, accion_clinica')
print()
print('  EWS — Estratificación (alineado con Fase 1 Business Understanding):')
print(f'    Bajo Riesgo   p < {EWS_BAJO}           → Monitoreo estándar')
print(f'    Riesgo Medio  {EWS_BAJO} ≤ p ≤ {EWS_MEDIO}    → Vigilancia intensiva (6-12h)')
print(f'    Alto Riesgo   p > {EWS_MEDIO}           → ALERTA INMEDIATA UCI')
print()
print('  Limitaciones:')
print('    - N=1000 fuente única. Validación externa en cohorte multicéntrica pendiente.')
print('    - Variables ausentes: tabaquismo, antecedentes familiares, IMC, LDL.')
print('    - Herramienta de apoyo — no reemplaza al clínico ni a biomarcadores.')
print('    - Ventana clínica objetivo: 6-12h (definida en Business Understanding).')

print(f'\n{s1}')
print(' '*20+'PROYECTO CARDIORISK COMPLETADO ✓')
print(' '*15+'IBM Data Science Professional Certificate')
print(s1)